# CausalLens: POPE Benchmark on Kaggle (2× NVIDIA T4 GPUs)

This notebook runs the **POPE (Polling-based Object Probing Evaluation)** benchmark on:
- **Qwen2-VL-7B-Instruct**
- **LLaVA-1.5-7B**

using **CausalLens** (Attention Adapter causal intervention) with **Greedy Decoding (`max_new_tokens=6`)** across all 3 splits: **Random**, **Popular**, and **Adversarial**.

In [ ]:
# ===== CELL 1: Environment Setup & Dependency Verification =====
# 1. Remove torchaudio to prevent CUDA driver mismatch
!pip uninstall -y -q torchaudio

# 2. Install necessary dependencies (Do NOT reinstall torch/torchvision to keep Kaggle CUDA drivers intact)
!pip install -q --upgrade \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece protobuf tiktoken \
    qwen-vl-utils pyyaml tqdm \
    huggingface_hub pandas

# 3. Verification Check
import torch, transformers, accelerate, qwen_vl_utils, sentencepiece
print(f"✅ PyTorch Version    : {torch.__version__}")
print(f"✅ CUDA Available     : {torch.cuda.is_available()} (Count: {torch.cuda.device_count()})")
print(f"✅ Transformers Vers  : {transformers.__version__}")
print(f"✅ Accelerate Vers    : {accelerate.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)")
    print(f"✅ BF16 Supported     : {torch.cuda.is_bf16_supported()}")

In [ ]:
# ===== CELL 2: HuggingFace Authentication =====
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace login successful via Kaggle Secrets!")
except Exception as e:
    hf_token = os.environ.get("HF_TOKEN", None)
    if hf_token:
        login(token=hf_token)
        print("✅ HuggingFace login successful via environment variable!")
    else:
        print("⚠️ HF_TOKEN not found. Public models will download normally, gated models require authentication.")

In [ ]:
# ===== CELL 3: Clone Repo & Verify Environment =====
import os, sys, torch

# Set repository directory
REPO_DIR = "/kaggle/working/CausalLens"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ntmy12/CausalLens.git {REPO_DIR}

# Add repo to sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Auto-detect COCO val2014 directory
from causallens_utils.coco_path_finder import find_coco_val2014_dir
COCO_DIR = find_coco_val2014_dir()
print(f"📁 Detected COCO val2014 Image Directory: {COCO_DIR}")

# Verify POPE annotation files
POPE_DIR = os.path.join(REPO_DIR, "experiments/data/POPE/coco")
for split in ["random", "popular", "adversarial"]:
    p_file = os.path.join(POPE_DIR, f"coco_pope_{split}.json")
    assert os.path.exists(p_file), f"Missing annotation: {p_file}"
print("📋 POPE Annotation Files: ✅ All 3 splits verified (Random, Popular, Adversarial)!")

In [ ]:
# ===== CELL 4: Configuration (CHOOSE MODEL) =====
# Select model to benchmark: 'qwen2vl' or 'llava'
MODEL = "qwen2vl"      # Options: 'qwen2vl' or 'llava'
SPLIT = "all"          # Options: 'random', 'popular', 'adversarial', 'all'

# CausalLens Hyperparameters
LAMBDA_CAUSAL = 0.15   # Intervention strength
GAMMA_MIX = 0.15       # Mixing ratio
LAYER_START = 10       # Start layer (inclusive)
LAYER_END = 20         # End layer (inclusive)
MAX_NEW_TOKENS = 6     # Mandatory POPE requirement (Greedy Decoding)
SEED = 42

print(f"Selected Model      : {MODEL.upper()}")
print(f"Selected Split      : {SPLIT}")
print(f"Intervention Params : lambda={LAMBDA_CAUSAL}, gamma={GAMMA_MIX}, layers=[{LAYER_START}, {LAYER_END}]")
print(f"Decoding Strategy   : Greedy (max_new_tokens={MAX_NEW_TOKENS})")

In [ ]:
# ===== CELL 5: Run CausalLens POPE Benchmark =====
!python kaggle_pope_runner.py \
    --model {MODEL} \
    --split {SPLIT} \
    --image_dir "{COCO_DIR}" \
    --pope_dir "experiments/data/POPE/coco" \
    --output_dir "results" \
    --lambda_causal {LAMBDA_CAUSAL} \
    --gamma_mix {GAMMA_MIX} \
    --layer_start {LAYER_START} \
    --layer_end {LAYER_END} \
    --max_new_tokens {MAX_NEW_TOKENS} \
    --seed {SEED}

In [ ]:
# ===== CELL 6: Display Summary & Metrics Table =====
import json, glob, pandas as pd

result_dirs = sorted(glob.glob(f"results/{MODEL}_pope_*"))
if result_dirs:
    latest_run = result_dirs[-1]
    rows = []
    for s in ["random", "popular", "adversarial"]:
        metrics_path = os.path.join(latest_run, s, "metrics.json")
        if os.path.exists(metrics_path):
            with open(metrics_path, "r") as f:
                m = json.load(f)
            rows.append({
                "Split": s.capitalize(),
                "Accuracy": f"{m.get('accuracy', 0.0)*100:.2f}%",
                "Precision": f"{m.get('precision', 0.0)*100:.2f}%",
                "Recall": f"{m.get('recall', 0.0)*100:.2f}%",
                "F1-Score": f"{m.get('f1', 0.0)*100:.2f}%",
                "Yes-Ratio": f"{m.get('yes_ratio', 0.0)*100:.2f}%",
                "Unknowns": m.get("unknowns", 0),
                "Total": m.get("total", 0),
            })

    if rows:
        df = pd.DataFrame(rows)
        print(f"\n{'='*76}")
        print(f"🏆 CausalLens POPE Benchmark Final Results - {MODEL.upper()}")
        print(f"📁 Output Folder: {latest_run}")
        print(f"{'='*76}")
        print(df.to_string(index=False))
        print(f"{'='*76}\n")
    else:
        print("⚠️ No split metrics.json found in run folder.")
else:
    print("⚠️ No results directory found matching results/{MODEL}_pope_*")